In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append('../')

import pandas as pd
import os
import subprocess
import zipfile
import matplotlib.pyplot as plt
import lightgbm as lgb
import numpy as np

from itertools import product
from sklearn.model_selection import GroupShuffleSplit
from itertools import chain
from sklearn.model_selection import GroupKFold
from src.utils import * 
from src.feature_engineering import *
from src.pipeline import *
from src.run import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
pd.reset_option('display.max_columns')
pd.set_option('display.max_columns', None)

In [ ]:
train = get_train_data()

Train shape: (18145372, 126)


In [ ]:
train = reduce_mem_usage(train)

Mem. usage decreased to 13843.82 Mb (19.2% reduction)


In [5]:
len(train.columns)

126

In [6]:
train = drop_constant_columns(train)

In [ ]:
col_names = train.columns.tolist()
duration_cols = []
for col in col_names:
    if 'duration' in col:
        duration_cols.append(col)
duration_cols

['legs0_duration',
 'legs0_segments0_duration',
 'legs0_segments1_duration',
 'legs0_segments2_duration',
 'legs0_segments3_duration',
 'legs1_duration',
 'legs1_segments0_duration',
 'legs1_segments1_duration',
 'legs1_segments2_duration']

In [10]:
train[duration_cols].head()

,legs0_duration,legs0_segments0_duration,legs0_segments1_duration,legs0_segments2_duration,legs0_segments3_duration,legs1_duration,legs1_segments0_duration,legs1_segments1_duration,legs1_segments2_duration
0,02:40:00,02:40:00,None,None,None,02:35:00,02:35:00,None,None
1,07:25:00,02:50:00,01:20:00,None,None,08:25:00,01:25:00,02:40:00,None
2,07:25:00,02:50:00,01:20:00,None,None,08:25:00,01:25:00,02:40:00,None
3,07:25:00,02:50:00,01:20:00,None,None,08:25:00,01:25:00,02:40:00,None
4,07:25:00,02:50:00,01:20:00,None,None,08:25:00,01:25:00,02:40:00,None


In [17]:
train = type_conversion(train)

In [88]:
train_df = train.copy()

In [89]:
col_names = train_df.columns.tolist()
col_names

['Id',
 'companyID',
 'corporateTariffCode',
 'frequentFlyer',
 'nationality',
 'isAccess3D',
 'isVip',
 'legs0_arrivalAt',
 'legs0_departureAt',
 'legs0_segments0_aircraft_code',
 'legs0_segments0_arrivalTo_airport_city_iata',
 'legs0_segments0_arrivalTo_airport_iata',
 'legs0_segments0_baggageAllowance_quantity',
 'legs0_segments0_baggageAllowance_weightMeasurementType',
 'legs0_segments0_cabinClass',
 'legs0_segments0_departureFrom_airport_iata',
 'legs0_segments0_flightNumber',
 'legs0_segments0_marketingCarrier_code',
 'legs0_segments0_operatingCarrier_code',
 'legs0_segments0_seatsAvailable',
 'legs0_segments1_aircraft_code',
 'legs0_segments1_arrivalTo_airport_city_iata',
 'legs0_segments1_arrivalTo_airport_iata',
 'legs0_segments1_baggageAllowance_quantity',
 'legs0_segments1_baggageAllowance_weightMeasurementType',
 'legs0_segments1_cabinClass',
 'legs0_segments1_departureFrom_airport_iata',
 'legs0_segments1_flightNumber',
 'legs0_segments1_marketingCarrier_code',
 'legs0_seg

In [90]:
train_df.head()

,Id,companyID,corporateTariffCode,frequentFlyer,nationality,isAccess3D,isVip,legs0_arrivalAt,legs0_departureAt,legs0_segments0_aircraft_code,legs0_segments0_arrivalTo_airport_city_iata,legs0_segments0_arrivalTo_airport_iata,legs0_segments0_baggageAllowance_quantity,legs0_segments0_baggageAllowance_weightMeasurementType,legs0_segments0_cabinClass,legs0_segments0_departureFrom_airport_iata,legs0_segments0_flightNumber,legs0_segments0_marketingCarrier_code,legs0_segments0_operatingCarrier_code,legs0_segments0_seatsAvailable,legs0_segments1_aircraft_code,legs0_segments1_arrivalTo_airport_city_iata,legs0_segments1_arrivalTo_airport_iata,legs0_segments1_baggageAllowance_quantity,legs0_segments1_baggageAllowance_weightMeasurementType,legs0_segments1_cabinClass,legs0_segments1_departureFrom_airport_iata,legs0_segments1_flightNumber,legs0_segments1_marketingCarrier_code,legs0_segments1_operatingCarrier_code,legs0_segments1_seatsAvailable,legs0_segments2_aircraft_code,legs0_segments2_arrivalTo_airport_city_iata,legs0_segments2_arrivalTo_airport_iata,legs0_segments2_baggageAllowance_quantity,legs0_segments2_baggageAllowance_weightMeasurementType,legs0_segments2_cabinClass,legs0_segments2_departureFrom_airport_iata,legs0_segments2_flightNumber,legs0_segments2_marketingCarrier_code,legs0_segments2_operatingCarrier_code,legs0_segments2_seatsAvailable,legs0_segments3_arrivalTo_airport_city_iata,legs0_segments3_arrivalTo_airport_iata,legs0_segments3_baggageAllowance_quantity,legs0_segments3_departureFrom_airport_iata,legs0_segments3_flightNumber,legs0_segments3_marketingCarrier_code,legs0_segments3_operatingCarrier_code,legs0_segments3_seatsAvailable,legs1_arrivalAt,legs1_departureAt,legs1_segments0_aircraft_code,legs1_segments0_arrivalTo_airport_city_iata,legs1_segments0_arrivalTo_airport_iata,legs1_segments0_baggageAllowance_quantity,legs1_segments0_baggageAllowance_weightMeasurementType,legs1_segments0_cabinClass,legs1_segments0_departureFrom_airport_iata,legs1_segments0_flightNumber,legs1_segments0_marketingCarrier_code,legs1_segments0_operatingCarrier_code,legs1_segments0_seatsAvailable,legs1_segments1_aircraft_code,legs1_segments1_arrivalTo_airport_city_iata,legs1_segments1_arrivalTo_airport_iata,legs1_segments1_baggageAllowance_quantity,legs1_segments1_baggageAllowance_weightMeasurementType,legs1_segments1_cabinClass,legs1_segments1_departureFrom_airport_iata,legs1_segments1_flightNumber,legs1_segments1_marketingCarrier_code,legs1_segments1_operatingCarrier_code,legs1_segments1_seatsAvailable,legs1_segments2_aircraft_code,legs1_segments2_arrivalTo_airport_city_iata,legs1_segments2_arrivalTo_airport_iata,legs1_segments2_baggageAllowance_quantity,legs1_segments2_baggageAllowance_weightMeasurementType,legs1_segments2_cabinClass,legs1_segments2_departureFrom_airport_iata,legs1_segments2_flightNumber,legs1_segments2_marketingCarrier_code,legs1_segments2_operatingCarrier_code,legs1_segments2_seatsAvailable,miniRules0_monetaryAmount,miniRules0_percentage,miniRules0_statusInfos,miniRules1_monetaryAmount,miniRules1_percentage,miniRules1_statusInfos,pricingInfo_isAccessTP,profileId,ranker_id,searchRoute,sex,taxes,totalPrice,selected,legs0_duration_minutes,legs0_segments0_duration_minutes,legs0_segments1_duration_minutes,legs0_segments2_duration_minutes,legs0_segments3_duration_minutes,legs1_duration_minutes,legs1_segments0_duration_minutes,legs1_segments1_duration_minutes,legs1_segments2_duration_minutes,requestDate_year,requestDate_month,requestDate_day,requestDate_hour
0,0,57323,<NA>,S7/SU/UT,36,False,False,2024-06-15T16:20:00,2024-06-15T15:40:00,YK2,KJA,KJA,1.0,0.0,1.0,TLK,216,KV,KV,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-07-09T14:20:00,2024-07-09T09:45:00,YK2,TLK,TLK,1.0,0.0,1.0,KJA,215.0,KV,KV,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2087645,98ce0dabf6964640b63079fbafd42cbe,T

In [91]:
rest_datetime_cols = ['legs0_arrivalAt', 'legs0_departureAt', 'legs1_arrivalAt', 'legs1_departureAt']

In [95]:
train_df = fix_datetime_columns(train_df, rest_datetime_cols)

In [100]:
train_df = searchRoute(train_df)

In [103]:
train_df['frequentFlyer_count'] = train_df['frequentFlyer'].str.split('/').str.len().fillna(0).astype('int8')
train_df['is_frequentFlyer'] = train_df['frequentFlyer'].str.len() > 0

In [105]:
train_df[['frequentFlyer_count', 'is_frequentFlyer']].tail()

,frequentFlyer_count,is_frequentFlyer
18146427,0,False
18146428,0,False
18146429,0,False
18146430,0,False
18146431,0,False


--- Baseline process ---